In [ ]:
import os
import json
import matplotlib.pyplot as plt
from collections import Counter
import numpy as np

plt.rcParams.update({
    "text.usetex": True,             # render all text with LaTeX
    "font.family": "serif",          # LaTeX serif font
    "font.serif": ["Computer Modern Roman"],  # explicitly CM
})

plt.rcParams.update({
    "font.size": 14+8,        # base font size
    "axes.titlesize": 16+8,   # title
    "axes.labelsize": 14+8,   # x and y labels
    "xtick.labelsize": 12+8,  # x tick labels
    "ytick.labelsize": 12+8,  # y tick labels
    "legend.fontsize": 12+8,  # legend
})

# Resize target
target_size = 800
max_size = 1333

# Use your resizing functions
def get_size_with_aspect_ratio(image_size, size, max_size=None):
    w, h = image_size
    if max_size is not None:
        min_original_size = float(min((w, h)))
        max_original_size = float(max((w, h)))
        if max_original_size / min_original_size * size > max_size:
            size = int(round(max_size * min_original_size / max_original_size))

    if (w <= h and w == size) or (h <= w and h == size):
        return (h, w)

    if w < h:
        ow = size
        oh = int(size * h / w)
    else:
        oh = size
        ow = int(size * w / h)

    return (oh, ow)

def get_size(image_size, size, max_size=None):
    if isinstance(size, (list, tuple)):
        return size[::-1]
    else:
        return get_size_with_aspect_ratio(image_size, size, max_size)

def clean_axes(ax):
    """Remove top and right spines for a cleaner look."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def compute_labelme_stats(json_dir):
    widths, heights = [], []
    n_images = 0

    for dirpath, _, filenames in os.walk(json_dir):
        for file in filenames:
            if file.endswith('.json'):
                path = os.path.join(dirpath, file)
                with open(path, "r") as f:
                    data = json.load(f)

                n_images += 1
                w, h = data.get("imageWidth"), data.get("imageHeight")
                if w and h:
                    widths.append(w)
                    heights.append(h)


    # Collect stats
    orig_res = []
    new_res = []
    ratios = []
    percent_loss = []
    scale_w = []
    scale_h = []

    for (w, h) in zip(widths, heights):
        oh, ow = get_size((w, h), target_size, max_size)
        orig = w * h
        new = ow * oh
        ratio = new / orig
        loss = 100 * (1 - ratio)
        
        orig_res.append(orig)
        new_res.append(new)
        ratios.append(ratio)
        percent_loss.append(loss)
        scale_w.append(ow / w)
        scale_h.append(oh / h)

    # --- Plot 1: Original vs Resized Resolution ---
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(1, 1, 1)
    # plt.scatter(orig_res, new_res, c="steelblue", alpha=0.7)
    # ax.set_xscale("log")
    # ax.set_yscale("log")
    # clean_axes(ax)
    # min_val = min(min(orig_res), min(new_res))
    # max_val = max(max(orig_res), max(new_res))
    
    # plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="darkgray", label="y=x")
    # plt.ylim(55e4, 11e5)
    # plt.xlabel("Original resolution (pixels)")
    # plt.ylabel("Resized resolution (pixels)")
    # plt.legend()

    # --- Plot 2: Histogram of ratios ---
    # ax = fig.add_subplot(2, 2, 2)
    # clean_axes(ax)
    # plt.hist(ratios, bins=10, color="steelblue", edgecolor="steelblue", alpha=0.7)
    # plt.xlabel("Resolution ratio (resized/original)")
    # plt.ylabel("\# images")
    # plt.axvline(1.0, color="darkgray", linestyle="--", label="No change")
    # plt.legend()

    # --- Plot 3: Histogram of percentage loss ---
    # ax = fig.add_subplot(2, 2, 3)
    # clean_axes(ax)
    percent_loss = [-pl for pl in percent_loss]
    plt.hist(percent_loss, bins=10, edgecolor="steelblue", alpha=0.7, color="steelblue")
    plt.xlabel("Resolution decrease (\%)",fontsize=36)
    plt.ylabel("\# images",fontsize=36)

    # # --- Plot 4: Scale factors ---
    # ax = fig.add_subplot(2, 2, 4)
    # clean_axes(ax)
    
    # plt.scatter(scale_w, scale_h, c="steelblue", alpha=0.7)
    # plt.xlabel("Width")
    # plt.ylabel("Height")
    # plt.axvline(1.0, color="darkgray", linestyle="--")
    # plt.axhline(1.0, color="darkgray", linestyle="--")
    clean_axes(ax)
    plt.tick_params(axis='both', which='major', labelsize=36)

    plt.tight_layout()
    plt.show()


compute_labelme_stats("data/EIDA/")

In [ ]:
import os
import json
import matplotlib.pyplot as plt
from collections import Counter
from matplotlib.ticker import LogFormatterExponent

plt.rcParams.update({
    "text.usetex": True,             # render all text with LaTeX
    "font.family": "serif",          # LaTeX serif font
    "font.serif": ["Computer Modern Roman"],  # explicitly CM
})

plt.rcParams.update({
    "font.size": 20,        # base font size
    "axes.titlesize": 16,   # title
    "axes.labelsize": 36,   # x and y labels
    "xtick.labelsize": 36,  # x tick labels
    "ytick.labelsize": 36,  # y tick labels
    "legend.fontsize": 14,  # legend
})

def clean_axes(ax):
    """Remove top and right spines for a cleaner look."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def compute_labelme_stats(json_dir):
    class_counter = Counter()
    widths, heights, aspect_ratios = [], [], []
    n_images = 0
    n_annots = 0

    for dirpath, _, filenames in os.walk(json_dir):
        for file in filenames:
            if file.endswith('.json'):
                path = os.path.join(dirpath, file)
                with open(path, "r") as f:
                    data = json.load(f)

                n_images += 1
                w, h = data.get("imageWidth"), data.get("imageHeight")
                if w and h and h > 0:
                    widths.append(w)
                    heights.append(h)
                    aspect_ratios.append(w / h)

                for shape in data.get("shapes", []):
                    class_counter[shape["label"]] += 1
                    n_annots += 1

    print(f"Images: {n_images}")
    print(f"Total annotations: {n_annots}")

    # --- Plot 1: Annotation counts per class ---
    plt.figure(figsize=(8,5))
    class_counter = dict(sorted(class_counter.items()))
    long = class_counter.pop("long")
    word = class_counter.pop("word")
    symbol = class_counter.pop("symbol")
    # class_counter = {"long": long, "word": word, "symbol": symbol, **class_counter}

    new_dict = {}
    other_keys = {}
    #class_counter.update({"unknown (?)": class_counter["?"]})
    #class_counter.pop("?")
    for k, v in class_counter.items():
        if v < 30:
            new_dict["others"] = new_dict.get("others", 0) + v
            other_keys.update({k: v})

    other_keys = dict(sorted(other_keys.items(), key=lambda item: item[1], reverse=True))    
    print(other_keys)
    print(new_dict["others"])
    print(len(other_keys))

    class_counter = {**new_dict, **{"long": long, "word": word, "symbol": symbol}, **class_counter} if "others" in new_dict else class_counter

    colors = []
    edge_colors = []
    for k in class_counter.keys():
        if k in other_keys.keys():
            colors.append('lightblue')  # faded color for small-count classes
            edge_colors.append('lightblue')
        elif k == "others":
            colors.append('lightblue')  # default color
            edge_colors.append('steelblue')
        else:
            colors.append('steelblue')
            edge_colors.append('steelblue')

    new_cls_counter = {}
    rest_cls_counter = {}
    for k, v in class_counter.items():
        if v >= 30:
            new_cls_counter[k] = v
        else:
            rest_cls_counter[k] = v
    plt.figure(figsize=(14, 10))  # wider figure
    colors = ['lightblue' if k=="others" else 'steelblue' for k in new_cls_counter.keys()]
    plt.bar(new_cls_counter.keys(), new_cls_counter.values(), color=colors, edgecolor="steelblue", alpha=0.7)
    print(new_cls_counter)
    #plt.axhline(30, color="gray", linestyle="--", label="threshold")
    #plt.title("Number of Annotations" )
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    # 0.47 is the top of the "others" bar
    # normalize bin sizes to relative heights
    total = sum(other_keys.values())
    heights = {k: v / total for k, v in other_keys.items()}

    x_pos = 0.03
    y_start = 0.44  # top position in Axes coordinates
    scale = 0.44     # total vertical space used (0–1)
    y = y_start

    for k, v in other_keys.items():
          h = heights[k] * scale
          if v <= 10:
            pass
        #   elif v <= 10:
        #     plt.text(x_pos, y-0.005, k, transform=plt.gca().transAxes, fontsize=7,
        #         ha='center', color='black')
          else:
            plt.text(x_pos, y-0.01, k, transform=plt.gca().transAxes, fontsize=20,
                            ha='center', color='black')
          # center the dashed line below each label
          plt.text(x_pos, y - h / 2, "------", transform=plt.gca().transAxes,
                    fontsize=19, ha='center', color='steelblue')
          y -= h  # move downward proportional to bin size


    plt.xticks(rotation=90)
    # Get current x-tick labels
    xticks = plt.gca().get_xticklabels()

    # Rotate only the first 4 labels
    for label in xticks[:4]:
        label.set_rotation(45)
        label.set_ha('right') 
    clean_axes(plt.gca())
    plt.margins(x=0.01)
    #plt.legend()
    plt.tight_layout()

    plt.show()

    plt.figure(figsize=(14, 10)) 
    plt.bar(rest_cls_counter.keys(), rest_cls_counter.values(), color="lightblue", edgecolor="lightblue", alpha=0.7)
    #plt.axhline(30, color="gray", linestyle="--", label="threshold")
    #plt.title("Number of Annotations")
    plt.xlabel("Class", fontsize=36)
    plt.ylabel("Count", fontsize=36)
    plt.grid(axis='y', linestyle='--', alpha=0.7)

    plt.xticks(rotation=90)
     
    clean_axes(plt.gca())
    plt.margins(x=0.01)
    #plt.legend()
    plt.tight_layout()

    plt.show()

    # --- Plot 2D: Aspect ratio histogram ---
    _, bins, _ = plt.hist(aspect_ratios, bins=20)
    plt.close() 
    import numpy as np
    # histogram on log scale. 
    # Use non-equal bin sizes, such that they look equal on log scale.
    logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))

    # 2. Box area distribution
    plt.figure(figsize=(14, 10))
    plt.hist(aspect_ratios, bins=logbins, color="steelblue", edgecolor="steelblue", alpha=0.7)
    plt.xscale("log")
    plt.xlabel("Aspect Ratio", fontsize=36)
    plt.ylabel("Frequency", fontsize=36)
    clean_axes(plt.gca())
    plt.tight_layout()
    plt.show()

compute_labelme_stats("data/EIDA")


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import math 

plt.rcParams.update({
    "text.usetex": True,             # render all text with LaTeX
    "font.family": "serif",          # LaTeX serif font
    "font.serif": ["Computer Modern Roman"],  # explicitly CM
})

plt.rcParams.update({
    "font.size": 20,        # base font size
    "axes.titlesize": 16,   # title
    "axes.labelsize": 36,   # x and y labels
    "xtick.labelsize": 36,  # x tick labels
    "ytick.labelsize": 36,  # y tick labels
    "legend.fontsize": 20,  # legend
})

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

import os
import json
import numpy as np
from shapely.geometry import Polygon

def analyze_labelme(json_dir, heatmap_res=100):
    ann_counts = []
    box_areas = []
    aspect_ratios = []
    labels = []

    # For heatmap
    heatmap = np.zeros((heatmap_res, heatmap_res), dtype=np.float32)
    for dirpath, _, filenames in os.walk(json_dir):
        for file in filenames:
            if file.endswith('.json'):
                path = os.path.join(dirpath, file)
                with open(path, "r") as f:
                    data = json.load(f) 

                shapes = data.get("shapes", [])
                ann_counts.append(len(shapes))

                img_w, img_h = data["imageWidth"], data["imageHeight"]


                for shape in shapes:
                    if "label" in shape.keys():
                        labels.append(shape["label"])
                    
                    pts = shape["points"]
                    try:
                        poly = Polygon(pts)
                    except Exception as e:
                        # Handle cases where the points might not form a valid simple polygon
                        print(f"Skipping invalid polygon data: {e}")
                        continue

                    # 2. Calculate the area of the polygon
                    poly_area = poly.area
                    
                    # Skip shapes with zero or effectively zero area (e.g., lines or points)
                    if poly_area <= 1e-6: # Using a small threshold for robustness
                        continue

                    # Store the polygon area, normalized by image area
                    box_areas.append(poly_area / (img_w * img_h))
                    x_coords = [p[0] for p in pts]
                    y_coords = [p[1] for p in pts]
                    xmin, xmax = min(x_coords), max(x_coords)
                    ymin, ymax = min(y_coords), max(y_coords)

                    bw, bh = xmax - xmin, ymax - ymin
                    if bw <= 0 or bh <= 0:
                        continue

                    # Store box area and aspect ratio
                    # box_areas.append((bw * bh)/(img_w * img_h))
                    aspect_ratios.append(bw / bh)

                    # Compute relative center (0-1) after resize
                    cx_rel = (xmin + xmax) / 2 / img_w
                    cy_rel = (ymin + ymax) / 2 / img_h

                    gx = int(np.floor(cx_rel * heatmap_res))
                    gy = int(np.floor(cy_rel * heatmap_res))
                    #gx = np.clip(gx, 0, heatmap_res - 1)
                    #gy = np.clip(gy, 0, heatmap_res - 1)

                    heatmap[gy, gx] += 1


    # ---- Stats ----
    print("Per-image annotation counts:")
    print(f"  Mean   = {np.mean(ann_counts):.2f}")
    print(f"  Median = {np.median(ann_counts):.2f}")
    print(f"  Range  = {np.min(ann_counts)} – {np.max(ann_counts)}")
    print(f" Total annotations: {sum(ann_counts)}")
    # ---- Plots ----
    # 1. Per-image annotation distribution
    plt.figure(figsize=(14, 10))
    bins = [0, 10, 20, 30, 40, 50, 60, 70, 75, float('inf')]
    plt.hist(ann_counts, bins=bins, color="steelblue", edgecolor="steelblue", alpha=0.7)
    #plt.xscale('log')  # if using log scale
    plt.xticks([0, 10, 20, 30, 40, 50, 60, 70], ['0','10','20','30','40','50','60','70+'])
    # Draw media n line
    plt.axvline(np.median(ann_counts), color="black", linestyle="--", label="Median")
    plt.legend()
    #plt.title("Per-Image Annotation Counts")
    plt.xlabel("Annotations per Image")
    plt.ylabel("Frequency")
    clean_axes(plt.gca())
    plt.tight_layout()
    plt.show()


    _, bins, _ = plt.hist(box_areas, bins=30)
    plt.close() 
    # histogram on log scale. 
    # Use non-equal bin sizes, such that they look equal on log scale.
    logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))

    # 2. Box area distribution
    plt.figure(figsize=(14, 10))
    plt.hist(box_areas, bins=logbins, color="steelblue", edgecolor="steelblue", log=True, alpha=0.7)
    #plt.title("Box Area Distribution")
    plt.xlabel("Polygon Area")
    plt.ylabel("Frequency")
    plt.xscale("log")
    clean_axes(plt.gca())
    plt.tight_layout()
    plt.show()

    # 3. Aspect ratio distribution of boxes
    _, bins, _ = plt.hist(aspect_ratios, bins=30)
    plt.close() 
    # histogram on log scale. 
    # Use non-equal bin sizes, such that they look equal on log scale.
    logbins = np.logspace(np.log10(bins[0]),np.log10(bins[-1]),len(bins))

    plt.figure(figsize=(14, 10))
    plt.hist(aspect_ratios, bins=logbins, color="steelblue", edgecolor="steelblue", alpha=0.7)
    plt.xlabel("Aspect Ratio")
    plt.xscale("log")
    plt.ylabel("Frequency")
    clean_axes(plt.gca())
    plt.tight_layout()
    plt.show()


    plt.figure(figsize=(14, 10))
    cmap = plt.cm.Blues.copy()
    cmap.set_under(color="white")  # cells with 0 count
    xticks = [t for t in plt.gca().get_xticks() if t != 0 and t != 1]
    yticks = [t for t in plt.gca().get_yticks() if t != 1 and t != 0]
    plt.gca().set_xticks(xticks)
    plt.gca().set_yticks(yticks)
    plt.imshow(
        np.flipud(heatmap),
        cmap=cmap,
        origin="upper",
        extent=[0, 1, 0, 1],
        aspect="auto",
        vmin=1e-5
    )
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.gca().invert_yaxis()

    plt.colorbar(label="Text Region Count")
    plt.tight_layout()
    plt.show()


# Example usage:
analyze_labelme("data/EIDA", heatmap_res=25)


In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def plot_annotations_per_image_from_labelme(json_dir, bins=20, threshold=60):
    ann_counts = []

    # --- Collect counts per image ---
    for root, dirs, files in os.walk(json_dir):
        for fname in files:
            if not fname.endswith(".json"):
                continue
            path = os.path.join(root, fname)
            try:
                with open(path, "r") as f:
                    data = json.load(f)
                shapes = data.get("shapes", [])
                ann_counts.append(len(shapes))
            except (json.JSONDecodeError, IOError) as e:
                print(f"Error reading {path}: {e}")

    if not ann_counts:
        print("No annotations found in directory:", json_dir)
        return

    # --- Compute stats on raw data ---
    ann_counts = np.array(ann_counts)
    mean_val = np.mean(ann_counts)
    median_val = np.median(ann_counts)
    min_val, max_val = np.min(ann_counts), np.max(ann_counts)

    print("Per-image annotation counts:")
    print(f"  Mean   = {mean_val:.2f}")
    print(f"  Median = {median_val:.2f}")
    print(f"  Range  = {min_val} – {max_val}")

    # --- Binning Logic: Clip values to the threshold ---
    # This forces everything 60 and above into the 60 bin
    plot_data = np.clip(ann_counts, 0, threshold)

    # --- Plot histogram ---
    plt.figure(figsize=(7, 5), dpi=300)
    
    # We keep your original bin count, but ensure the range ends at threshold
    n, bins_edges, patches = plt.hist(
        plot_data, 
        bins=bins, 
        range=(0, threshold), 
        color="steelblue", 
        edgecolor="steelblue", 
        alpha=0.7
    )

    plt.axvline(
        x=min(median_val, threshold), 
        color="black", 
        linestyle="--", 
        linewidth=2, 
        label=f"Median: {median_val:.1f}"
    )

    plt.xlabel("Annotations per Image")
    plt.ylabel("Frequency")

    # --- Legend and Axis Formatting ---
    plt.legend() 

    # --- Update X-axis labels to show "60+" ---
    ax = plt.gca()
    current_ticks = ax.get_xticks()
    # Create labels, replacing the threshold value with "60+"
    new_labels = [str(int(t)) if t < threshold else f"{threshold}+" for t in current_ticks]
    ax.set_xticklabels(new_labels)

    clean_axes(ax)
    plt.tight_layout()
    plt.show()

# Example usage:
plot_annotations_per_image_from_labelme("/home/sonat/OCR_HDV/data/EIDA/", bins=10, threshold=50)

In [ ]:
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patheffects

def draw_polygons_with_arrows(image_path, json_path, save_path=None):
    # Load image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Load Labelme JSON
    with open(json_path, "r") as f:
        data = json.load(f)

    plt.figure(figsize=(12, 12))
    plt.imshow(img_rgb)
    ax = plt.gca()

    for shape in data["shapes"]:
        pts = np.array(shape["points"], dtype=np.int32)
        K = len(pts)

        # Draw polygon
        poly = plt.Polygon(pts, fill=False, edgecolor="green", linewidth=2)
        ax.add_patch(poly)

        # Baseline points (first half)
        baseline_pts = pts[:K // 2]

        # Get midpoint of baseline
        start = baseline_pts[0]
        end = baseline_pts[-1]
        mid = ((start[0] + end[0]) / 2, (start[1] + end[1]) / 2)

        # Compute orientation angle
        dx, dy = end[0] - start[0], end[1] - start[1]
        angle = np.degrees(np.arctan2(dy, dx))

        label = shape.get("label", "")
        if label:
            cx, cy = pts[-1, 0], pts[-1, 1]
            txt_label = ax.text(
                cx, cy - 2, f"{label}",
                fontsize=16, color="black", ha="center", va="bottom",
                    bbox=dict(facecolor="white", alpha=0.4, edgecolor="none", boxstyle="round,pad=0.3")

            )

        # Place arrow (use a text symbol like "→")
        if label != "symbol":
            arrow_text = "➔"  # you can change to "➡️" or "→"
            txt = ax.text(
                mid[0], mid[1], arrow_text,
                fontsize=16, color="green",
                rotation=angle, rotation_mode="anchor",
                ha="center", va="center"
            )
            # Add outline for visibility
            txt.set_path_effects([patheffects.withStroke(linewidth=3, foreground="darkgreen")])

    plt.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt
from collections import Counter
from matplotlib.ticker import LogFormatterExponent

#plt.rcParams.update({
#    "text.usetex": True,             # render all text with LaTeX
#    "font.family": "serif",          # LaTeX serif font
#    "font.serif": ["Computer Modern Roman"],  # explicitly CM
#})

plt.rcParams.update({
    "font.size": 14,        # base font size
    "axes.titlesize": 16,   # title
    "axes.labelsize": 14,   # x and y labels
    "xtick.labelsize": 12,  # x tick labels
    "ytick.labelsize": 12,  # y tick labels
    "legend.fontsize": 12,  # legend
})

def clean_axes(ax):
    """Remove top and right spines for a cleaner look."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)




def list_unknown(json_dir):
    hist_unk = []
    for fname in os.listdir(json_dir):
        if not fname.endswith(".json"):
            continue
        path = os.path.join(json_dir, fname)
        with open(path, "r") as f:
            data = json.load(f)

        for shape in data.get("shapes", []):
            if shape["label"]=="?":
                hist_unk.append(fname)

    hist_unk = list(set(hist_unk))  # unique filenames
    for f in hist_unk:
        print(f)
        # Plot the images that have "?" with all annotations
        draw_polygons_with_arrows(image_path=os.path.join(json_dir, f.replace(".json", ".jpg")), 
                                json_path=os.path.join(json_dir, f))


list_unknown("data/diagramlatsplit/data/")
